# Exploración del Titanic

Ejecuta `make pipeline` desde la raíz antes de correr este cuaderno. Silver tiene las 891 filas etiquetadas; Bronze conserva la fuente original para auditar faltantes. Las celdas exploran asociación, no causalidad.

In [ ]:
import importlib

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from flows.common.paths import BRONZE, BRONZE_LABELED_NAME, GOLD, REPORTS, SILVER, SILVER_NAME

sns.set_theme(style="whitegrid", context="notebook")
df = pd.read_parquet(SILVER / SILVER_NAME)
bronze = pd.read_csv(BRONZE / BRONZE_LABELED_NAME)
agg = importlib.import_module("flows.03_aggregation")
print(f"Silver: {df.shape[0]} pasajeros x {df.shape[1]} variables")

## Pasos 1–5: estructura y calidad

La fuente contiene 891 pasajeros etiquetados; `shape`, tipos, muestra y estadísticos descriptivos verifican lo recibido. La tabla y el mapa muestran los faltantes antes de imputar.

In [ ]:
display(bronze.shape)
display(pd.DataFrame({"variable": bronze.columns, "tipo": bronze.dtypes.astype(str).values}))
display(bronze.head(10))
display(bronze.describe(include="all").T)
missing = pd.DataFrame({"faltantes": bronze.isna().sum(), "porcentaje": bronze.isna().mean().mul(100)})
display(missing.sort_values("faltantes", ascending=False))
REPORTS.mkdir(parents=True, exist_ok=True)
missing.to_csv(REPORTS / "data_quality.csv")
plt.figure(figsize=(12, 5))
sns.heatmap(bronze.isna(), cbar=False, yticklabels=False)
plt.title("Mapa de valores faltantes en Bronze")
plt.xlabel("Variable"); plt.tight_layout(); plt.show()

## Pasos 6–10: género, edad y tipo de persona

`Person` marca como niño a quien tiene menos de 16 años. Cuando la edad falta, conserva el sexo; esto puede subcontar niños. KDE describe edades observadas y no prueba causalidad.

In [ ]:
display(bronze["Sex"].value_counts().rename("pasajeros"))
display(pd.crosstab(bronze["Pclass"], bronze["Sex"], margins=True))
display(bronze["Age"].describe())
display(df["Person"].value_counts(dropna=False))
display(pd.crosstab(df["Pclass"], df["Person"]))
sns.histplot(bronze["Age"].dropna(), bins=30)
plt.title("Distribución de edad conocida"); plt.xlabel("Edad"); plt.show()
sns.kdeplot(data=bronze.dropna(subset=["Age"]), x="Age", hue="Sex", fill=True, common_norm=False)
plt.title("Densidad de edad por sexo"); plt.show()
sns.kdeplot(data=bronze.dropna(subset=["Age"]), x="Age", hue="Pclass", common_norm=False)
plt.title("Densidad de edad por clase"); plt.show()

## Pasos 11–15: cabina, puerto y familia

`X` representa cabina desconocida; no elimina pasajeros. La cubierta `T` se omite solo en el gráfico para evitar que un grupo unitario distorsione la lectura. La tabla de puerto se normaliza por fila.

In [ ]:
display(df["TCabin"].value_counts().sort_index())
known_decks = df.loc[df["TCabin"] != "X", "TCabin"]
sns.countplot(x=known_decks, order=sorted(known_decks.unique()))
plt.title("Pasajeros por cubierta conocida"); plt.xlabel("Cubierta"); plt.show()
clean_decks = known_decks[known_decks != "T"]
sns.countplot(x=clean_decks, order=sorted(clean_decks.unique()))
plt.title("Cubiertas conocidas (T excluida solo del gráfico)"); plt.show()
sns.countplot(data=df, x="Embarked", hue="Pclass")
plt.title("Puerto de embarque y clase"); plt.show()
display(pd.crosstab(df["Embarked"], df["Pclass"], normalize="index"))
display(df["Alone"].value_counts(normalize=True).rename("proporción"))
sns.countplot(data=df, x="Alone")
plt.title("Viaja solo o con familia"); plt.show()

## Pasos 16–20: supervivencia

La media de `Survived` es la tasa. Los intervalos de confianza reflejan incertidumbre; las diferencias son asociaciones observadas en este naufragio.

In [ ]:
display(df["Survivor"].value_counts())
print(f"Tasa general: {df['Survived'].mean():.1%} (n={len(df)})")
sns.catplot(data=df, x="Pclass", y="Survived", kind="point", height=4, aspect=1.4)
plt.title("Tasa de supervivencia por clase"); plt.ylabel("Tasa"); plt.show()
sns.catplot(data=df, x="Pclass", y="Survived", hue="Person", hue_order=["child", "female", "male"], kind="point", height=4, aspect=1.5)
plt.title("Supervivencia por clase y tipo de persona"); plt.ylabel("Tasa"); plt.show()
sns.lmplot(data=bronze.dropna(subset=["Age"]), x="Age", y="Survived", height=4)
plt.title("Edad y supervivencia (tendencia descriptiva)"); plt.show()
for group in ["Pclass", "Sex"]:
    sns.lmplot(data=bronze.dropna(subset=["Age"]), x="Age", y="Survived", hue=group, x_bins=[10, 20, 40, 60, 80], height=4)
    plt.title(f"Tasa por edad agrupada y {group}"); plt.show()
plt.scatter(bronze["Age"], bronze["Fare"], alpha=.5)
plt.title("Edad y tarifa"); plt.xlabel("Edad"); plt.ylabel("Tarifa"); plt.show()
sns.scatterplot(data=bronze, x="Age", y="Fare", hue="Pclass", alpha=.5, palette="viridis")
plt.title("Edad y tarifa por clase"); plt.show()

## Pasos 21–29: imputación, asociaciones y estadística

Silver conserva `AgeOriginal` y el indicador `AgeImputada`; los códigos `N*` son útiles para correlaciones, pero imponen un orden artificial. Chi-cuadrado detecta asociación, no tamaño causal.

In [ ]:
print("Embarked faltantes en Bronze:", bronze["Embarked"].isna().sum(), "; moda:", bronze["Embarked"].mode().iloc[0])
print("Age faltantes en Bronze:", bronze["Age"].isna().sum(), "; imputadas en Silver:", df["AgeImputada"].sum())
print("Desviación de edad observada/imputada:", df["AgeOriginal"].std(), df["Age"].std())
display(df["TTitulo"].value_counts())
display(pd.crosstab(df["TTitulo"], df["Sex"]))
display(df[["Sex", "NSex", "Embarked", "NEmbarked", "TCabin", "NCabin", "TTitulo", "NTitulo"]].head())
cols = ["Survived", "Pclass", "NSex", "Age", "SibSp", "Parch", "Fare", "NEmbarked", "NCabin", "NTitulo"]
corr = df[cols].corr()
plt.figure(figsize=(9, 7)); sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlaciones lineales"); plt.show()
display(corr["Survived"].sort_values())
for name, table in agg.group_stats(df).items():
    print(name); display(table)
chi = agg.chi2_sex_survival(df)
print("Chi-cuadrado sexo × supervivencia:", chi)
print("Un p pequeño indica asociación estadística, no magnitud ni causalidad.")

## Pasos 30–38: comparaciones, Gold y atípicos

La imputación reduce artificialmente la dispersión de edades; por eso los boxplots usan `AgeOriginal`. Los atípicos son candidatos a revisión, no errores automáticos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, column in zip(axes.flat, ["Sex", "Pclass", "Embarked", "Survived"]):
    sns.boxplot(data=df, x=column, y="AgeOriginal", ax=ax)
    ax.set_title(f"Edad original por {column}")
plt.tight_layout(); plt.show()
sns.histplot(df["Fare"], kde=True, bins=40)
plt.title("Distribución de tarifa"); plt.show()
print("Tarifa: media", df["Fare"].mean(), "mediana", df["Fare"].median(), "asimetría", df["Fare"].skew())
display(agg.survival_by_class_sex(df))
display(agg.survival_by_embarked(df))
display(agg.survival_by_embarked_class(df))
display(agg.survival_by_fare_quartile_class(df))
display(agg.survival_by_family(df))
display(agg.outlier_summary(df))
display(df.nlargest(10, "Fare")[["PassengerId", "Name", "Pclass", "Fare", "Survived"]])
print("Las tablas Parquet persistidas están en", GOLD)